# TimesFM Zero-Shot Evaluation

Benchmarks TimesFM 2.5 (zero-shot) against a per-LSOA historical-mean baseline, using the same sample and train/test regimes as the Bayesian model so results are directly comparable.

**Benchmark sample:** 5 forces × 5 crimes = 25 force×crime combos per regime  
**Forces:** Cheshire, Lincolnshire, Merseyside, Metropolitan, West Midlands  
**Crimes:** Anti-social behaviour, Criminal damage and arson, Drugs, Possession of weapons, Vehicle crime  

| Regime | Train | Test |
|---|---|---|
| Pre-pandemic | 2012–2018 | 2019 |
| Post-pandemic | 2022–2023 | 2024 |
| Combined non-pandemic | 2012–2019 + 2022–2024 | 2025 |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import timesfm
from itertools import product
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

## 1. Load and prepare data

In [ ]:
FORCES = {
    'Cheshire Constabulary':         'Cheshire',
    'Lincolnshire Police':            'Lincolnshire',
    'Merseyside Police':              'Merseyside',
    'Metropolitan Police Service':    'Metropolitan',
    'West Midlands Police':           'West Midlands',
}
CRIMES = [
    'Anti-social behaviour',
    'Criminal damage and arson',
    'Drugs',
    'Possession of weapons',
    'Vehicle crime',
]

raw = pd.read_parquet('../data/processed/crimes_clean_dedup_all_years.parquet')
raw = raw[raw['Falls within'].isin(FORCES) & raw['Crime type'].isin(CRIMES)].copy()
raw['force'] = raw['Falls within'].map(FORCES)
raw['month'] = pd.to_datetime(raw['Month'])
raw = raw[raw['month'].dt.year >= 2012]

# Aggregate to LSOA × force × crime × month incident counts
counts = (
    raw.groupby(['LSOA code', 'force', 'Crime type', 'month'])
    .size()
    .reset_index(name='count')
)
print(f"Rows: {len(counts):,}  |  LSOAs: {counts['LSOA code'].nunique():,}")
print(f"Date range: {counts['month'].min().date()} → {counts['month'].max().date()}")

## 2. Regime definitions

In [ ]:
REGIMES = {
    'Pre-pandemic': {
        'train_years': set(range(2012, 2019)),
        'test_years':  {2019},
    },
    'Post-pandemic': {
        'train_years': {2022, 2023},
        'test_years':  {2024},
    },
    'Combined non-pandemic': {
        'train_years': set(range(2012, 2020)) | {2022, 2023, 2024},
        'test_years':  {2025},
    },
}
REGIME_ORDER = ['Pre-pandemic', 'Post-pandemic', 'Combined non-pandemic']

## 3. Load TimesFM model

In [ ]:
REPO_ID = 'google/timesfm-2.5-200m-pytorch'

print(f'Loading TimesFM from {REPO_ID} ...')
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(REPO_ID)

# Compile with a forecast config (max 12 months horizon, context up to 512 months)
forecast_cfg = timesfm.ForecastConfig(
    max_context=512,
    max_horizon=128,  # must be a multiple of output_patch_len (128)
    per_core_batch_size=32,
    infer_is_positive=True,  # counts are non-negative
    normalize_inputs=True,
)
tfm.compile(forecast_cfg)
print('Model ready.')

## 4. Evaluation helpers

In [ ]:
# metric helpers (defined once, reused across the notebook)
def mae(pred, actual):
    return float(np.mean(np.abs(pred - actual)))

def rmse(pred, actual):
    return float(np.sqrt(np.mean((pred - actual) ** 2)))


def build_panel(df_slice, train_months, test_months):
    """Return (train_array, test_array) each shape (n_lsoas, n_months) with zero-fill."""
    all_months = sorted(set(train_months) | set(test_months))
    pivot = df_slice.pivot_table(
        index='LSOA code', columns='month', values='count', aggfunc='sum', fill_value=0
    )
    for m in all_months:
        if m not in pivot.columns:
            pivot[m] = 0
    pivot = pivot[sorted(all_months)]
    train_arr = pivot[sorted(train_months)].values.astype(float)
    test_arr  = pivot[sorted(test_months)].values.astype(float)
    return train_arr, test_arr


def evaluate_combo(force, crime, regime_name, regime_cfg, counts_df, model):
    """Returns metrics dict for one force × crime combo in a given regime."""
    slice_df = counts_df[(counts_df['force'] == force) & (counts_df['Crime type'] == crime)]

    train_months = sorted(
        [m for m in slice_df['month'].unique() if m.year in regime_cfg['train_years']]
    )
    test_months = sorted(
        [m for m in slice_df['month'].unique() if m.year in regime_cfg['test_years']]
    )

    if not train_months or not test_months:
        return None

    train_arr, test_arr = build_panel(slice_df, train_months, test_months)
    n_lsoas, n_test = train_arr.shape[0], len(test_months)

    # historical mean baseline
    baseline_preds = np.tile(train_arr.mean(axis=1, keepdims=True), (1, n_test))

    # timesfm zero-shot forecast. the combined regime has a 2020-2021 gap,
    # so the contiguous training series per lsoa is passed as context.
    # forecast() takes (horizon, list_of_1d_arrays).
    inputs = [row for row in train_arr]
    point_forecast, _ = model.forecast(horizon=n_test, inputs=inputs)
    tfm_preds = np.clip(np.array(point_forecast)[:, :n_test], 0, None)

    return {
        'force':          force,
        'crime':          crime,
        'regime':         regime_name,
        'n_lsoas':        n_lsoas,
        'n_test_months':  n_test,
        'baseline_mae':   mae(baseline_preds, test_arr),
        'baseline_rmse':  rmse(baseline_preds, test_arr),
        'model_mae':      mae(tfm_preds, test_arr),
        'model_rmse':     rmse(tfm_preds, test_arr),
    }

## 5. Run evaluation across all regimes

In [ ]:
results = []
force_labels = list(FORCES.values())
total = len(REGIMES) * len(force_labels) * len(CRIMES)

for done, (regime_name, force, crime) in enumerate(
    product(REGIME_ORDER, force_labels, CRIMES), start=1
):
    regime_cfg = REGIMES[regime_name]
    print(f'[{done:02d}/{total}] {regime_name} | {force} | {crime}', end='  ')
    res = evaluate_combo(force, crime, regime_name, regime_cfg, counts, tfm)
    if res:
        results.append(res)
        print(f"baseline={res['baseline_mae']:.3f}  TimesFM={res['model_mae']:.3f}")
    else:
        print('SKIPPED (no data)')

results_df = pd.DataFrame(results)
results_df['delta_mae']  = results_df['baseline_mae']  - results_df['model_mae']
results_df['delta_rmse'] = results_df['baseline_rmse'] - results_df['model_rmse']
results_df['rmae']       = results_df['model_mae'] / results_df['baseline_mae']
results_df['model_wins'] = results_df['model_mae'] < results_df['baseline_mae']
print(f'\nDone. {len(results_df)} combos evaluated.')

## 6. Headline results by regime

In [ ]:
headline = (
    results_df.groupby('regime')
    .agg(
        baseline_mae  = ('baseline_mae',  'mean'),
        model_mae     = ('model_mae',     'mean'),
        baseline_rmse = ('baseline_rmse', 'mean'),
        model_rmse    = ('model_rmse',    'mean'),
        delta_mae     = ('delta_mae',     'mean'),
        rmae          = ('rmae',          'mean'),
        win_rate      = ('model_wins',    'mean'),
    )
    .reset_index()
)
headline['delta_mae_pct'] = (headline['delta_mae'] / headline['baseline_mae'] * 100).round(1)
headline['win_rate_pct']  = (headline['win_rate'] * 100).round(0).astype(int)
headline['regime'] = pd.Categorical(headline['regime'], categories=REGIME_ORDER, ordered=True)
headline = headline.sort_values('regime').reset_index(drop=True)

display(
    headline[['regime', 'baseline_mae', 'model_mae', 'delta_mae', 'delta_mae_pct',
              'baseline_rmse', 'model_rmse', 'rmae', 'win_rate_pct']]
    .round(3)
    .rename(columns={
        'regime': 'Regime',
        'baseline_mae': 'Baseline MAE', 'model_mae': 'TimesFM MAE',
        'delta_mae': 'Δ MAE', 'delta_mae_pct': 'Δ MAE %',
        'baseline_rmse': 'Baseline RMSE', 'model_rmse': 'TimesFM RMSE',
        'rmae': 'RMAE', 'win_rate_pct': 'Win rate %',
    })
    .set_index('Regime')
)

## 7. Results by force

In [ ]:
by_force = (
    results_df.groupby(['regime', 'force'])
    .agg(delta_mae=('delta_mae', 'mean'), win_rate=('model_wins', 'mean'))
    .reset_index()
)
by_force['win_pct'] = (by_force['win_rate'] * 100).round(0).astype(int)

force_order = ['Cheshire', 'Lincolnshire', 'Merseyside', 'Metropolitan', 'West Midlands']

# Delta MAE pivot
delta_pivot = by_force.pivot_table(index='force', columns='regime', values='delta_mae', aggfunc='first')
delta_pivot = delta_pivot[[c for c in REGIME_ORDER if c in delta_pivot.columns]]
delta_pivot.index = pd.CategoricalIndex(delta_pivot.index, categories=force_order, ordered=True)
delta_pivot = delta_pivot.sort_index()

# Win rate pivot
win_pivot = by_force.pivot_table(index='force', columns='regime', values='win_pct', aggfunc='first')
win_pivot = win_pivot[[c for c in REGIME_ORDER if c in win_pivot.columns]]
win_pivot.index = pd.CategoricalIndex(win_pivot.index, categories=force_order, ordered=True)
win_pivot = win_pivot.sort_index()

print('=== Δ MAE by force (positive = TimesFM beats baseline) ===')
display(delta_pivot.round(3))
print('\n=== Win rate % by force ===')
display(win_pivot)

## 8. Results by crime type

In [ ]:
by_crime = (
    results_df.groupby(['regime', 'crime'])
    .agg(
        baseline_mae=('baseline_mae', 'mean'),
        model_mae=('model_mae', 'mean'),
        delta_mae=('delta_mae', 'mean'),
        rmae=('rmae', 'mean'),
        win_rate=('model_wins', 'mean'),
    )
    .reset_index()
)
by_crime['win_pct'] = (by_crime['win_rate'] * 100).round(0).astype(int)

print('=== Combined non-pandemic regime ===')
combined_crime = (
    by_crime[by_crime['regime'] == 'Combined non-pandemic']
    .sort_values('delta_mae', ascending=False)
)
display(
    combined_crime[['crime', 'baseline_mae', 'model_mae', 'delta_mae', 'rmae', 'win_pct']]
    .round(3).set_index('crime')
)

print('\n=== Cross-regime Δ MAE by crime type ===')
cross = by_crime.pivot_table(index='crime', columns='regime', values='delta_mae', aggfunc='first')
cross = cross[[c for c in REGIME_ORDER if c in cross.columns]]
display(cross.round(3))

## 9. TimesFM vs Bayesian side-by-side

In [ ]:
# Bayesian results from the briefing document
bayesian = pd.DataFrame([
    {'regime': 'Pre-pandemic',          'bayes_mae': 1.150, 'bayes_rmae': 1.021, 'bayes_win_pct': 40},
    {'regime': 'Post-pandemic',         'bayes_mae': 1.126, 'bayes_rmae': 1.029, 'bayes_win_pct': 60},
    {'regime': 'Combined non-pandemic', 'bayes_mae': 1.116, 'bayes_rmae': 0.950, 'bayes_win_pct': 56},
])

comp = (
    headline[['regime', 'baseline_mae', 'model_mae', 'rmae', 'win_rate_pct']]
    .rename(columns={'model_mae': 'tfm_mae', 'rmae': 'tfm_rmae', 'win_rate_pct': 'tfm_win_pct'})
    .merge(bayesian, on='regime')
)
comp['regime'] = pd.Categorical(comp['regime'], categories=REGIME_ORDER, ordered=True)
comp = comp.sort_values('regime').reset_index(drop=True)

display(
    comp[['regime', 'baseline_mae',
          'tfm_mae', 'tfm_rmae', 'tfm_win_pct',
          'bayes_mae', 'bayes_rmae', 'bayes_win_pct']]
    .round(3)
    .rename(columns={
        'regime': 'Regime',
        'baseline_mae': 'Baseline MAE',
        'tfm_mae': 'TimesFM MAE', 'tfm_rmae': 'TimesFM RMAE', 'tfm_win_pct': 'TFM Win%',
        'bayes_mae': 'Bayesian MAE', 'bayes_rmae': 'Bayes RMAE', 'bayes_win_pct': 'Bayes Win%',
    })
    .set_index('Regime')
)

## 10. Visualisations

In [ ]:
# --- Bar chart: MAE comparison across regimes ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('MAE by Regime — TimesFM vs Bayesian vs Baseline', fontsize=13, fontweight='bold')

for ax, regime in zip(axes, REGIME_ORDER):
    row_tfm   = comp[comp['regime'] == regime].iloc[0]
    labels = ['Baseline', 'TimesFM', 'Bayesian']
    vals   = [row_tfm['baseline_mae'], row_tfm['tfm_mae'], row_tfm['bayes_mae']]
    colors = ['#9E9E9E', '#2196F3', '#FF7043']
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', linewidth=0.8, width=0.55)
    ax.set_title(regime, fontsize=10)
    ax.set_ylabel('Mean MAE')
    ax.set_ylim(0, max(vals) * 1.25)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + max(vals) * 0.02,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/timesfm_mae_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Heatmap: Δ MAE by force × regime ---
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    delta_pivot.round(3), annot=True, fmt='.3f', cmap='RdYlGn',
    center=0, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Δ MAE  (positive = TimesFM better than baseline)'}
)
ax.set_title('TimesFM: Δ MAE by Force × Regime', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../outputs/timesfm_force_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Heatmap: Δ MAE by crime × regime ---
crime_delta = by_crime.pivot_table(index='crime', columns='regime', values='delta_mae', aggfunc='first')
crime_delta = crime_delta[[c for c in REGIME_ORDER if c in crime_delta.columns]]

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    crime_delta.round(3), annot=True, fmt='.3f', cmap='RdYlGn',
    center=0, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Δ MAE  (positive = TimesFM better than baseline)'}
)
ax.set_title('TimesFM: Δ MAE by Crime Type × Regime', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../outputs/timesfm_crime_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Save results

In [ ]:
results_df.to_csv('../outputs/timesfm_results_all_combos.csv', index=False)
comp.to_csv('../outputs/timesfm_vs_bayesian.csv', index=False)
print('Saved to ../outputs/')